# ANN — next-hour CO₂ and NOx forecasting (Plant Bowen)

This notebook is the interactive version of `train_ann.py`. It runs the same experiment step by step, so you can inspect the data, the model and the results between stages.

It does not redefine anything. The data pipeline comes from `src/data_prep.py`, the figures from `src/plots.py`, and the network and metrics from `train_ann.py`. As a result the notebook and the script always train the same model on the same data. To change a hyperparameter, edit it in `train_ann.py` and re-run this notebook.

**Kernel:** select the project's `.venv` interpreter.

## 1. Setup

In [ ]:
import os
import sys
import json
import time
from pathlib import Path

os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

# Works whether the notebook runs from the project root (VS Code default here)
# or from inside notebooks/.
PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
for p in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from IPython.display import Image, display

import data_prep
import plots
import train_ann as ann

# Where outputs go. These are the same folders the script writes to, so
# re-running this notebook overwrites the script's ANN results and figures.
RESULTS_DIR = PROJECT_ROOT / "results"
plots.FIG_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR.mkdir(exist_ok=True)
plots.FIG_DIR.mkdir(exist_ok=True)

keras.utils.set_random_seed(ann.SEED)
print(f"TensorFlow {tf.__version__} | seed {ann.SEED} | "
      f"lookback {data_prep.LOOKBACK} h | horizon {data_prep.HORIZON} h")
print(f"Hyperparameters: hidden={ann.HIDDEN_UNITS}, dropout={ann.DROPOUT}, "
      f"lr={ann.LEARNING_RATE}, batch={ann.BATCH_SIZE}, max epochs={ann.EPOCHS}")


def show(name: str, width: int = 760) -> None:
    # plots.py saves figures to disk instead of drawing them inline, so we
    # display the saved PNG.
    display(Image(filename=str(plots.FIG_DIR / f"{name}.png"), width=width))


def slug(target: str) -> str:
    return "co2" if "CO2" in target else "nox"

## 2. Data

`data_prep.prepare()` loads the CSV, adds cyclical features, keeps operating hours only, splits the series into continuous blocks at every gap, builds 24-hour windows that never cross a gap, splits by time (70/15/15) and fits the scalers on the training set only.

It is called once per pollutant because each pollutant has its own target scaler.

In [ ]:
data = {target: data_prep.prepare(target) for target in data_prep.TARGETS}

In [ ]:
d = data[data_prep.TARGETS[0]]
print("Input features:", len(d["feature_names"]))
for i, name in enumerate(d["feature_names"]):
    print(f"  {i:2d}  {name}")
print("\nX_train shape (windows, hours, features):", d["X_train"].shape)

### Dataset figures

These describe the data rather than a model, so they are produced once.

In [ ]:
df = d["df"]
plots.fig_target_timeline(df, data_prep.TARGETS)
plots.fig_block_lengths(df, d["lookback"])
plots.fig_correlation(
    df, data_prep.PLANT_COLS + ["SO2 Mass (lbs)"] + data_prep.WEATHER_COLS + ["WD50M"]
)
plots.fig_daily_seasonal_profile(df, data_prep.TARGETS)
plots.fig_target_distribution(df, data_prep.TARGETS)
plots.fig_split_diagram(d)

for name in ["01_data_timeline", "02_block_lengths", "03_correlation_matrix",
             "04_daily_seasonal_profile", "05_target_distributions",
             "06_chronological_split"]:
    show(name)

## 3. Model

The ANN flattens each 24 × 17 window into one vector of 408 numbers, so it has no built-in notion of time order. That makes it the baseline architecture for the comparison.

It predicts the **change** since the last hour, not the absolute level, and is trained with Huber loss. Both choices are shared with the Transformer, so the comparison isolates architecture. `src/data_prep.py` explains the change target.

In [ ]:
example = ann.build_ann(d["lookback"], d["X_train"].shape[2])
example.summary()

## 4. Training

There is one model per pollutant. `ann.train()` sets the seed, builds the network, and fits it on the scaled hour-to-hour change (see `data_prep.change_targets()`) with Huber loss. Early stopping watches validation loss and restores the best weights, and the learning rate halves when validation loss plateaus.

In [ ]:
models, histories, change_scalers, train_seconds = {}, {}, {}, {}

for target in data_prep.TARGETS:
    print(f"\n{'#' * 70}\n#  {ann.MODEL_NAME} — {target}\n{'#' * 70}")
    (models[target], histories[target],
     change_scalers[target], train_seconds[target]) = ann.train(data[target])
    print(f"Finished in {train_seconds[target]:.1f}s after "
          f"{len(histories[target].history['loss'])} epochs")

## 5. Evaluation on the test set

This is the first time the test set is used. `ann.forecast()` adds the predicted change to the last observed hour, so predictions are in real units (short tons and lbs) before scoring.

**Persistence** is the naive baseline: it predicts that next hour equals this hour. A model has to beat it to be worth anything.

In [ ]:
predictions, metrics = {}, {}
rows = []
for target in data_prep.TARGETS:
    dt = data[target]
    predictions[target] = ann.forecast(models[target], dt, change_scalers[target])
    metrics[target] = ann.evaluate(dt["y_test"], predictions[target],
                                   dt["persistence_test"])
    for key in ["mae", "rmse", "mape", "r2"]:
        m, b = metrics[target]["model"][key], metrics[target]["persistence"][key]
        rows.append({
            "target": target, "metric": key.upper(),
            ann.MODEL_NAME: m, "Persistence": b,
            "improvement vs persistence (%)":
                None if key == "r2" else (b - m) / b * 100,
        })

pd.DataFrame(rows).set_index(["target", "metric"]).style.format(
    {ann.MODEL_NAME: "{:,.3f}", "Persistence": "{:,.3f}",
     "improvement vs persistence (%)": "{:+.1f}"}, na_rep="—")

### Model figures

There are six figures per pollutant: the training curves, two weeks of test-set predictions, predicted vs actual, residuals, the comparison with the baseline, and errors broken down by hour and month.

In [ ]:
for target in data_prep.TARGETS:
    dt, pred, m = data[target], predictions[target], metrics[target]
    plots.fig_training_history(histories[target], target, ann.MODEL_NAME)
    plots.fig_prediction_timeseries(dt["ts_test"], dt["y_test"], pred, target, ann.MODEL_NAME)
    plots.fig_scatter(dt["y_test"], pred, target, ann.MODEL_NAME, m["model"]["r2"])
    plots.fig_residuals(dt["y_test"], pred, target, ann.MODEL_NAME)
    plots.fig_baseline_comparison(m, target, ann.MODEL_NAME)
    plots.fig_error_breakdown(dt["ts_test"], dt["y_test"], pred, target, ann.MODEL_NAME)

In [ ]:
FIGURE_KINDS = [("07", "training_history"), ("08", "timeseries"), ("09", "scatter"),
                ("10", "residuals"), ("11", "vs_baseline"), ("12", "error_breakdown")]
for target in data_prep.TARGETS:
    print(f"\n=== {target} ===")
    for number, kind in FIGURE_KINDS:
        show(f"{number}_{slug(target)}_{ann.MODEL_NAME.lower()}_{kind}")

## 6. Save

The saved files have the same names and format as the script's output, so any later comparison across architectures can read them unchanged.

In [ ]:
results = []
for target in data_prep.TARGETS:
    dt, s = data[target], slug(target)
    models[target].save(RESULTS_DIR / f"ann_{s}.keras")
    np.savez_compressed(
        RESULTS_DIR / f"ann_{s}_predictions.npz",
        ts=dt["ts_test"], y_true=dt["y_test"], y_pred=predictions[target],
        persistence=dt["persistence_test"],
    )
    results.append({
        "target": target,
        "model": ann.MODEL_NAME,
        "epochs_run": len(histories[target].history["loss"]),
        "train_seconds": round(train_seconds[target], 1),
        "parameters": int(models[target].count_params()),
        "n_train": int(len(dt["y_train"])),
        "n_val": int(len(dt["y_val"])),
        "n_test": int(len(dt["y_test"])),
        "metrics": metrics[target],
    })

(RESULTS_DIR / "ann_metrics.json").write_text(json.dumps(results, indent=2))
print("Saved models, predictions and metrics to", RESULTS_DIR)